In [1]:
# system
import os
import sys
ROOT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(ROOT_PATH)

import pandas as pd
import numpy as np

# personalized modules
from src.utils.dataset import get_raw_transactions

from sklearn.preprocessing import StandardScaler, RobustScaler, FunctionTransformer

#models
from sklearn.pipeline import Pipeline
from src.unsupervised_model import AMLAnomalyEnsemble, IsolationForestModel, HDBScanModel

#preprocess
from src.features import FeatureGenerator
from src.preprocess import PreprocessTransactions

2026-03-31 22:41:00.065 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-31 22:41:00.065 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-31 22:41:00.069 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [2]:
data_path = f"{ROOT_PATH}\\src\\data\\output\\raw_transactions.csv"
raw_df = get_raw_transactions(data_path)
raw_df = raw_df.sample(n=1000)

In [3]:
raw_df.head()

,timestamp,transaction_date,sender_customer,receiver_customer,amount
3164482,2022-09-07 19:12:00,2022-09-07,81426D901_54348,814768E11_152416,63.397884
2589620,2022-09-06 14:33:00,2022-09-06,805A507F0_111631,81163F9C0_117660,1322.979245
475560,2022-09-01 17:02:00,2022-09-01,8042975C0_3051,8058C3B50_113685,0.330000
1863024,2022-09-05 02:03:00,2022-09-05,8078BBDB0_22775,80B62F2E0_25382,999.803448
79751,2022-09-01 00:08:00,2022-09-01,80E1996C0_137218,80E9D45B0_239604,241793.077289


In [4]:
feature_generator = FeatureGenerator()
feature_generator.fit(raw_df)
feature_df = feature_generator.transform(raw_df)

In [5]:
feature_columns = [col for col in feature_df.columns if col != 'customer_id']
X = feature_df[feature_columns]

In [6]:
log_transformer = FunctionTransformer(np.log1p, validate=False)
identity_transformer = FunctionTransformer(lambda x: x, validate=False)

pipeline_iforest = Pipeline([
    ("identity", identity_transformer),  # no log
    ("scaler", RobustScaler()),
    ("model", IsolationForestModel())
])

pipeline_hdbscan = Pipeline([
    ("log", log_transformer),
    ("scaler", RobustScaler()),
    ("model", HDBScanModel())
])

models = [
    pipeline_iforest,
    pipeline_hdbscan
]

ensemble = AMLAnomalyEnsemble(models=models)
ensemble.fit(X)

c:\Users\ferna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [ ]:
feature_df

,customer_id,transaction_count_received,transaction_count_sent,total_amount_received,total_amount_sent,median_amount_received,median_amount_sent,std_amount_received,std_amount_sent,max_amount_received,...,transaction_count_received_30d,transaction_count_received_90d,sent_received_ratio,transaction_direction_ratio,total_amount_30d_ratio,total_amount_7d_ratio,total_amount_90d_ratio,transaction_count_30d_ratio,transaction_count_7d_ratio,transaction_count_90d_ratio
0,100428660_70,0.0,43.0,0.000000,7.607821e+05,0.000000,1015.240000,0.0,7.130881e+04,0.000000,...,0,0,7.607821e+05,43.0,8.940734e+10,6.713747e+11,0.0,11000000.0,32000000.0,0.0
1,1004286A8_70,0.0,22.0,0.000000,1.353932e+07,0.000000,951.135180,0.0,2.832964e+06,0.000000,...,0,0,1.353932e+07,22.0,8.296610e+10,1.345635e+13,0.0,8000000.0,14000000.0,0.0
2,1004286F0_70,0.0,5.0,0.000000,2.303135e+07,0.000000,2084.691291,0.0,1.028241e+07,0.000000,...,0,0,2.303135e+07,5.0,0.000000e+00,2.303135e+13,0.0,0.0,5000000.0,0.0
3,100428738_70,0.0,6.0,0.000000,1.033417e+04,0.000000,1977.971653,0.0,8.378209e+02,0.000000,...,0,0,1.033417e+04,6.0,0.000000e+00,1.033417e+10,0.0,0.0,6000000.0,0.0
4,100428780_70,0.0,4.0,0.000000,1.028550e+05,0.000000,558.241651,0.0,5.067799e+04,0.000000,...,0,0,1.028550e+05,4.0,0.000000e+00,1.028550e+11,0.0,0.0,4000000.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1856,81489F901_55653,1.0,0.0,127317.701928,0.000000e+00,127317.701928,0.000000,0.0,0.000000e+00,127317.701928,...,0,0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0
1857,8148B8BC1_55878,1.0,0.0,10921.739322,0.000000e+00,10921.739322,0.000000,0.0,0.000000e+00,10921.739322,...,0,0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0
1858,81492A981_225,1.0,0.0,264.512425,0.000000e+00,264.512425,0.000000,0.0,0.000000e+00,264.512425,...,0,0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0
1859,814935741_220,1.0,0.0,4115.756608,0.000000e+00,4115.756608,0.000000,0.0,0.000000e+00,4115.756608,...,0,0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0


In [11]:
scores = ensemble.score(X)

c:\Users\ferna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\pipeline.py:62: FutureWarning:

This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.



In [10]:
import plotly.express as px

In [12]:
fig = px.violin(
    y=scores,
    box=False,
    points=False,
    title="Combined Risk Score Density"
)
fig.show()

In [ ]:
robust_preprocess = PreprocessTransactions()
robust_preprocess.fit(feature_df)

standard_preprocess = PreprocessTransactions(scaler=StandardScaler())
standard_preprocess.fit(feature_df)

X_log_robust_scaled = robust_preprocess.transform(feature_df)
X_log_standard_scaled = standard_preprocess.transform(feature_df)